# Customer Churn Prediction — From Data to Model

This notebook walks through the full ML pipeline:
1. Load the Kaggle e-commerce churn dataset
2. Exploratory Data Analysis
3. Preprocessing (imputation + encoding)
4. Hyperparameter Tuning with GridSearchCV
5. Model Evaluation
6. Save the trained model

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
df = pd.read_csv("../datasets/data_ecommerce_customer_churn.csv")
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print(f"\nNull counts:")
print(df.isnull().sum())
print(f"\nBasic stats:")
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df["Churn"].value_counts()
churn_counts.index = ["Active", "Churned"]
churn_counts.plot(kind="bar", ax=axes[0], color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Churn Class Balance")
axes[0].set_ylabel("Customers")

df["PreferedOrderCat"].value_counts().plot(kind="barh", ax=axes[1])
axes[1].set_title("Preferred Order Category")
axes[1].set_xlabel("Customers")

plt.tight_layout()
plt.show()
print(f"Churn rate: {df['Churn'].mean():.1%}")

In [ ]:
numeric_cols = ["Tenure", "WarehouseToHome", "NumberOfDeviceRegistered",
                "SatisfactionScore", "NumberOfAddress", "DaySinceLastOrder", "CashbackAmount"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 4, i % 4]
    df.boxplot(column=col, by="Churn", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")

axes[1, 3].axis("off")
plt.suptitle("Feature Distributions by Churn", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Preprocessing

- Impute nulls (Tenure, WarehouseToHome, DaySinceLastOrder) with median
- One-hot encode categoricals (PreferedOrderCat, MaritalStatus)

In [ ]:
impute_medians = {}
for col in ["Tenure", "WarehouseToHome", "DaySinceLastOrder"]:
    median_val = df[col].median()
    impute_medians[col] = median_val
    df[col] = df[col].fillna(median_val).astype(int)

print("Imputation medians:", impute_medians)
print("Remaining nulls:", df.isnull().sum().sum())

In [ ]:
target = "Churn"

feature_df = df.drop(columns=[target])
feature_df = pd.get_dummies(feature_df, columns=["PreferedOrderCat", "MaritalStatus"], drop_first=True)

X = feature_df
y = df[target]

print(f"Feature matrix shape: {X.shape}")
print(f"Feature columns: {list(X.columns)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {len(X_train):,} customers")
print(f"Test set:     {len(X_test):,} customers")

## 4. Hyperparameter Tuning with GridSearchCV

We search over key RandomForest hyperparameters using 5-fold cross-validation, scoring on F1 (due to class imbalance at ~17% churn).

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print(f"\nBest F1 score (CV): {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Test accuracy: {(y_pred == y_test).mean():.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Active", "Churned"]))

## 5. Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Active", "Churned"],
            yticklabels=["Active", "Churned"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind="barh", title="Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("Top 5 features:")
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f"  {feat}: {imp:.4f}")

## 6. Save Model

We save the model along with the feature column names and imputation medians, so the Streamlit app can reproduce the exact same feature transformation.

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

artifact = {
    "model": best_model,
    "feature_columns": list(X.columns),
    "impute_medians": impute_medians,
}
joblib.dump(artifact, "../models/churn_model.pkl")

print("Model artifact saved to models/churn_model.pkl")
print(f"  Feature columns ({len(artifact['feature_columns'])}): {artifact['feature_columns']}")
print(f"  Impute medians: {artifact['impute_medians']}")

# Quick verification
loaded = joblib.load("../models/churn_model.pkl")
sample = X_test.iloc[:1]
prob = loaded["model"].predict_proba(sample)[0]
print(f"\nSample prediction — Active: {prob[0]:.1%}, Churned: {prob[1]:.1%}")
print("Done!")